In [ ]:
import pandas as pd

online_df = pd.read_csv('../data/ford-catalogue/online_catalogue.csv')

online_df= online_df[["call_type", "clan", "pod", "filename", "audio_fp", "spect_fp"]]

In [9]:
import re
def extract_call_number(call_name):
    """Remove any 'i' or 'v' characters after the call number"""
    if pd.isna(call_name):
        return ""
    # Match pattern like N01, N16, etc. - everything before the roman numerals
    match = re.match(r'([A-Z]\d+)', str(call_name))
    return match.group(1) if match else str(call_name)

online_df['call_number'] = online_df['call_type'].apply(extract_call_number)

In [10]:
online_df

,call_type,clan,pod,filename,audio_fp,spect_fp,call_number
0,N01i,A,A1,N01i-A1-1,data/ford-catalogue/Northern Resident/media/A_...,data/ford-catalogue/spects/N01i-A1-1.png,N01
1,N01i,A,A1,N01i-A1-2,data/ford-catalogue/Northern Resident/media/A_...,data/ford-catalogue/spects/N01i-A1-2.png,N01
2,N01i,A,A1,N01i-A1-3,data/ford-catalogue/Northern Resident/media/A_...,data/ford-catalogue/spects/N01i-A1-3.png,N01
3,N01i,A,A1,N01i-A1-4,data/ford-catalogue/Northern Resident/media/A_...,data/ford-catalogue/spects/N01i-A1-4.png,N01
4,N01ii,A,B1,N01ii-B1-1,data/ford-catalogue/Northern Resident/media/A_...,data/ford-catalogue/spects/N01ii-B1-1.png,N01
...,...,...,...,...,...,...,...
197,N51,R,R1,N51-R1-1,data/ford-catalogue/Northern Resident/media/R_...,data/ford-catalogue/spects/N51-R1-1.png,N51
198,N51,R,R1,N51-R1-2,data/ford-catalogue/Northern Resident/media/R_...,data/ford-catalogue/spects/N51-R1-2.png,N51
199,N52,R,R1,N52-R1-1,data/ford-catalogue/Northern Resident/media/R_...,data/ford-catalogue/spects/N52-R1-1.png,N52
200,N53,R,R1,N53-R1-1,data/ford-catalogue/Northern Resident/media/R_...,data/ford-catalogue/spects/N53-R1-1.png,N53


In [17]:
import numpy as np

# Set random seed for reproducibility
np.random.seed(4)

# Initialize annotator column
online_df['annotator'] = None

# Group by call_type
for call_type, group in online_df.groupby('call_type'):
    indices = group.index.tolist()
    num_rows = len(indices)
    
    if num_rows == 1:
        # Single row: assign to either annotator (alternating or random)
        online_df.loc[indices[0], 'annotator'] = np.random.choice(['emmanuel', 'maddie'])
    else:
        # More than one row: ensure each annotator gets at least one
        # First row to emmanuel
        online_df.loc[indices[0], 'annotator'] = 'emmanuel'
        # Second row to maddie
        online_df.loc[indices[1], 'annotator'] = 'maddie'
        
        # Split the remaining rows 50/50
        if num_rows > 2:
            remaining_indices = indices[2:]
            # Shuffle for randomness
            np.random.shuffle(remaining_indices)
            # Split in half
            mid_point = len(remaining_indices) // 2
            
            # First half to emmanuel
            for idx in remaining_indices[:mid_point]:
                online_df.loc[idx, 'annotator'] = 'maddie'
            
            # Second half to maddie
            for idx in remaining_indices[mid_point:]:
                online_df.loc[idx, 'annotator'] = 'emmanuel'

online_df

,call_type,clan,pod,filename,audio_fp,spect_fp,call_number,annotator
0,N01i,A,A1,N01i-A1-1,data/ford-catalogue/Northern Resident/media/A_...,data/ford-catalogue/spects/N01i-A1-1.png,N01,emmanuel
1,N01i,A,A1,N01i-A1-2,data/ford-catalogue/Northern Resident/media/A_...,data/ford-catalogue/spects/N01i-A1-2.png,N01,maddie
2,N01i,A,A1,N01i-A1-3,data/ford-catalogue/Northern Resident/media/A_...,data/ford-catalogue/spects/N01i-A1-3.png,N01,emmanuel
3,N01i,A,A1,N01i-A1-4,data/ford-catalogue/Northern Resident/media/A_...,data/ford-catalogue/spects/N01i-A1-4.png,N01,maddie
4,N01ii,A,B1,N01ii-B1-1,data/ford-catalogue/Northern Resident/media/A_...,data/ford-catalogue/spects/N01ii-B1-1.png,N01,emmanuel
...,...,...,...,...,...,...,...,...
197,N51,R,R1,N51-R1-1,data/ford-catalogue/Northern Resident/media/R_...,data/ford-catalogue/spects/N51-R1-1.png,N51,emmanuel
198,N51,R,R1,N51-R1-2,data/ford-catalogue/Northern Resident/media/R_...,data/ford-catalogue/spects/N51-R1-2.png,N51,maddie
199,N52,R,R1,N52-R1-1,data/ford-catalogue/Northern Resident/media/R_...,data/ford-catalogue/spects/N52-R1-1.png,N52,emmanuel
200,N53,R,R1,N53-R1-1,data/ford-catalogue/Northern Resident/media/R_...,data/ford-catalogue/spects/N53-R1-1.png,N53,emmanuel


In [18]:
# Verify the distribution
print("Overall annotator counts:")
print(online_df['annotator'].value_counts())
print("\nAnnotator distribution by call_type (for call_types with >1 row):")
multi_row_types = online_df.groupby('call_type').filter(lambda x: len(x) > 1)
print(multi_row_types.groupby(['call_type', 'annotator']).size().unstack(fill_value=0))

Overall annotator counts:
annotator
emmanuel    107
maddie       95
Name: count, dtype: int64

Annotator distribution by call_type (for call_types with >1 row):
annotator  emmanuel  maddie
call_type                  
N01i              2       2
N01ii             2       2
N01iii            1       1
N02               3       2
N03               3       2
N04               4       4
N05i              6       5
N05ii             3       3
N07i              5       4
N07ii             1       1
N07iii            2       2
N07iv             1       1
N08i              4       4
N08ii             1       1
N08iii            1       1
N08iv             1       1
N09i              2       2
N09iii            2       2
N11               2       1
N12               7       6
N13               2       1
N16i              2       1
N16ii             2       2
N16iv             1       1
N17               2       1
N18               2       1
N20               2       2
N23i              2       2

In [19]:
online_df["annotated"] = False

In [ ]:
online_df.to_csv("../data/ford-catalogue/online_catalogue_annotated.csv", index=False)